In [5]:
import torch
from base_model import KATSum
from transformers import LongT5ForConditionalGeneration,AutoTokenizer
from kg_embedder import KGEncoder
from datasets import load_dataset,load_from_disk
from torch.utils.data import Dataset, DataLoader

In [6]:
MODEL_NAME = "google/long-t5-tglobal-base"
BATCH_SIZE = 4
GRAD_ACCUM = 16
SRC_MAX_LEN = 4096
TGT_MAX_LEN = 512
NUM_SIDECAR_LAYERS = 12
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [9]:
train_dataset = load_from_disk("./dataset/pubmed_with_triples_rebalanced/train")
val_dataset = load_from_disk("./dataset/pubmed_with_triples_rebalanced/validation")
test_dataset = load_from_disk("./dataset/pubmed_with_triples_rebalanced/test")

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = LongT5ForConditionalGeneration.from_pretrained(MODEL_NAME)

kg_embedder = KGEncoder(
    encoder=base_model.encoder,
    tokenizer=tokenizer,
    hidden_dim=base_model.config.d_model,
    device=DEVICE,
)

model = KATSum(
    base_model=base_model,
    kg_embedder=kg_embedder,
    num_sidecar_layers=NUM_SIDECAR_LAYERS,
    device=DEVICE,
)

model.parameter_count()

/home/shreesh/Documents/project_repositories/nlp-kg-summarization/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Number of decoder layers for KATSUM model: 12
Sidecar indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Number of sidecar layers: 12
Total parameters: 247,596,684
Trainable parameters: 9,228 (0.0%)
Frozen parameters: 247,587,456 (100.0%)
KG Embedder params: 0
Sidecar layer params: 84,962,316
Num sidecar layers: 12


In [12]:
generation_config = {
    "max_new_tokens": 150,  # Give it room to finish the thought
    "num_beams": 4,  # Standard sweet spot for quality vs speed
    "min_length": 40,  # Prevent "one-sentence" lazy summaries
    "length_penalty": 2.0,  # Higher (>1.0) encourages longer, complete sentences
    "no_repeat_ngram_size": 3,  # Prevents the "participants... participants" loop
    "early_stopping": True,  # Stop as soon as all beams hit the </s> token
    "repetition_penalty": 2.5,  # Heavily discourages copying the same phrases
}

tokens = tokenizer(val_dataset[0]["article"], return_tensors="pt", padding=True)

output_ids = model.generate_summary(
    input_ids=tokens["input_ids"],
    attention_mask=tokens["attention_mask"],
    triples=val_dataset[0]["rebel_triples"],
    **generation_config
)

tokenizer.decode(token_ids=output_ids[0], skip_special_tokens=True)

/home/shreesh/Documents/project_repositories/nlp-kg-summarization/.venv/lib/python3.11/site-packages/transformers/modeling_utils.py:949: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


'these data for case group comprised of height, weight, age at the onset of disease, 24 h urinary protein excretion, serum albumin, blood urea - creatine - lyso - meth - polymorphic ne - and - micro - biochemical studies have been performed to evaluate the role of cystatin - angiotensin – raf iv in predicting response to steroid therapy, we found that patients who achieved remission with or without fibrosis had a lower gfr (p = 0.016 compared with healthy controls '